# Daily Data on Casualties (Mediazona)

In [1]:
import requests
import time 
import pandas as pd
from random import uniform
from bs4 import BeautifulSoup
from urllib.parse import urljoin

In [ ]:
base_url = "https://200.zona.media"

# 1. Get all regions
def get_regions():
    regions_url = urljoin(base_url, "все_регионы.html")
    response = requests.get(regions_url)
    response.encoding = 'utf-8'
    soup = BeautifulSoup(response.text, 'html.parser')
    return [a['href'] for a in soup.select('ul.tiles a[href$=".html"]')]

# 2. Get military branches for each region
def get_military_branches(region_url):
    time.sleep(uniform(0.5, 1.5))
    response = requests.get(region_url)
    response.encoding = 'utf-8'
    soup = BeautifulSoup(response.text, 'html.parser')
    return [urljoin(base_url, a['href']) 
            for a in soup.select('ul.tiles a[href*="/"]')]

# 3. Get personal profiles from each military branch
def get_profiles(branch_url):
    time.sleep(uniform(0.5, 1.5))
    response = requests.get(branch_url)
    response.encoding = 'utf-8'
    soup = BeautifulSoup(response.text, 'html.parser')
    
    tiles_div = soup.select_one('div.tiles')
    if not tiles_div:
        return []  
    return [urljoin(base_url, a['href']) 
            for a in tiles_div.find_all('a', href=True)]



In [ ]:
all_profiles = []
regions = get_regions()

for region in regions:
    region_url = urljoin(base_url, region)
    branches = get_military_branches(region_url)    
    for branch_url in branches:
        profiles = get_profiles(branch_url)
        all_profiles.extend(profiles)


In [ ]:
extracted_data = []

for url in all_profiles:
    response = requests.get(url)
    response.encoding = 'utf-8'
    
    soup = BeautifulSoup(response.text, 'html.parser')
    dates_div = soup.select_one('div.card__dates')
    if dates_div:
        dates_text = dates_div.get_text(strip=True)
        if "—" in dates_text:
            birth_date, death_date = dates_text.split("—")
            birth_date = birth_date.strip()
            death_date = death_date.split("(")[0].strip()  # Remove age part
    parts = url.replace("https://200.zona.media/", "").split("/")
    region = parts[0].replace("_", " ")
    branch = parts[1].replace("_", " ")
    name = parts[2].replace("_", " ").replace(".html", "")
    extracted_data.append({
        'region': region,
        'branch': branch,
        'name' : name,
        'birth_date' : birth_date,
        'death_date' : death_date 
        })

df = pd.DataFrame(extracted_data)

In [15]:
df.to_csv("data/daily.csv")

In [ ]:
date_pattern = r'\d{2}\.\d{2}\.\d{4}'
result = [item for item in list if not re.match(date_pattern, str(item))]
print(result)

In [ ]:
df = pd.read_csv("data/daily.csv")

['03.03.2022',
 '06.12.2023',
 '19.11.2024',
 '04.07.2022',
 '03.05.2022',
 '22.03.2022',
 '17.07.2022',
 '16.09.2022',
 '21.12.2022',
 '03.06.2023',
 '21.05.2023',
 '07.05.2022',
 '04.10.2023',
 '03.10.2023',
 '25.08.2023',
 '17.09.2024',
 '05.11.2024',
 '22.09.2022',
 '23.04.2022',
 '25.04.2022',
 '05.03.2022',
 '07.06.2022',
 '11.12.2022',
 '21.08.2022',
 '21.04.2022',
 '21.05.2022',
 '19.09.2022',
 '12.08.2022',
 '19.05.2022',
 '07.09.2022',
 '10.06.2022',
 '24.05.2022',
 '27.04.2022',
 '29.08.2022',
 '14.06.2022',
 '07.03.2022',
 '11.03.2022',
 '05.02.2023',
 '29.11.2022',
 '01.07.2022; 29.06.2022',
 '24.02.2022',
 '?',
 '06.02.2023',
 '19.06.2023',
 '30.06.2023',
 '31.08.2023',
 '30.09.2023',
 '27.08.2022',
 '04.11.2023',
 '06.06.2022',
 '16.11.2023',
 '30.12.2023',
 '03.12.2023',
 '27.02.2024',
 '26.02.2024',
 '21.03.2022',
 '10.03.2024',
 '21.03.2024',
 '19.01.2024',
 '12.12.2023',
 '29.04.2024',
 '10.05.2024',
 '02.04.2024',
 '10.07.2023',
 '26.09.2024',
 '14.09.2024',
 '27.04

In [ ]:
# import re
unique_death_dates = df["death_date"].unique().tolist()
date_pattern = [r'\d{2}\.\d{2}\.\d{4}', r'\d{2}\.\d{4}'] 
result = [item for item in unique_death_dates
          if not (
              re.match(date_pattern[0], str(item))
              or re.match(date_pattern[1], str(item))
        )]
print(result)

['?', '2022', '??.03.2022', '2024', '2023', '??.08.2024', '22–25.07.2024', '9.07.2024', 'не иранее 24.05.2024', '2025', '??.04.2023', '18/21.08.2022', '20.5.2023', '04.03. 2024', '?7.??.2024', '??.04.2022', '??.10.2023', '05/06.03.2022', 'не ранее 16.08.2023', 'март 2023', '2.04.2023', '8.02.2024', '??.12.2024', '7.12.2023', '25.5.2023', '??.11.2022', 'март 2024', '21-22.07.2024', 'не ранее 12.01.2024', 'не ранее 15.09.2024', '2.07.2023', '2024/2025', 'не ранее 28.03.2024', '??.02.2023', 'не ранее 12.12.2023', '02 12.2022', '??.??.2024', 'не ранее 22.11.2024', 'не ранее 16.10.2024', '??.05.2024', '??.01.2023', '8.08.2023', '8.05.2022', '9.01.2024', '2?.09.2022', '5.08.2022', '4.03.2022', 'не ранее 13.09.2024', 'не ранее 24.07.2024', 'не ранее 07.04. 2024', 'не ранее  21.10.2024', 'февраль 2025', '??.??.2023', '??.01.2025', 'не ранее 07.10.2023', '??.09.2023', '?.02.2024', 'июнь 2024', 'не ранее 01.06.2024', 'не ранее 19.09.2024', 'не ранее 15.07.2024', 'весна 2023', 'не ранее 02.2024',

In [31]:
def split_names(full_name):
    try:
        surname = full_name.split(" ")[0]
        first_name = full_name.split(" ")[1]
    except IndexError:
        surname, first_name = full_name, ''
    return surname, first_name
    


In [32]:
surnames = []
first_names = []
for i in range(len(df)):
    surname, first_name = split_names(df.loc[i, "name"])
    surnames.append(surname)
    first_names.append(first_name)

In [ ]:
names = pd.DataFrame(list(zip(first_names, surnames)), columns=["first_name", "last_name"])
names.to_csv("ruEthnicNamesPublic/data/names.csv")